# Notebook 02: Matrix Factorization

## Why This Matters

Matrix factorization won the Netflix Prize. In 2009, Simon Funk's blog post describing Funk SVD — a method for factorizing a ratings matrix using stochastic gradient descent on observed entries only — became one of the most influential algorithmic ideas in recommender systems history. The winning Netflix Prize team used an ensemble of matrix factorization models. Today, matrix factorization in various forms (ALS, BPR, neural MF) underpins recommendation at Spotify, Amazon, and virtually every large-scale personalization system. Understanding it deeply is table stakes for any ML engineer working on recommendations.

## Background

### SVD: Why It Doesn't Directly Apply

Singular Value Decomposition decomposes any matrix $R = U \Sigma V^T$. In theory, you could use the top-$k$ singular values to get a low-rank approximation. In practice, SVD requires a *complete* matrix — you'd have to fill in missing entries first (with zeros or means), which introduces massive noise. Filling missing entries as zeros would tell the model that users rate unrated movies as 0 stars, which is factually wrong.

### Funk SVD: The Key Insight

Simon Funk's insight: **only optimize on observed ratings**. Decompose $R \approx P Q^T$ where:
- $P \in \mathbb{R}^{m \times k}$ — user embedding matrix ($m$ users, $k$ latent factors)
- $Q \in \mathbb{R}^{n \times k}$ — item embedding matrix ($n$ items, $k$ latent factors)
- $\hat{r}_{ui} = \mathbf{p}_u \cdot \mathbf{q}_i$ — predicted rating

Minimize the regularized squared error over *observed* ratings only:

$$\mathcal{L} = \sum_{(u,i) \in \Omega} (r_{ui} - \mathbf{p}_u \cdot \mathbf{q}_i)^2 + \lambda(\|\mathbf{p}_u\|^2 + \|\mathbf{q}_i\|^2)$$

where $\Omega$ is the set of observed (user, item) pairs.

### ALS: Alternating Least Squares

Instead of SGD, ALS alternates between two convex subproblems:
1. **Fix $Q$, solve for $P$**: Each user's embedding $\mathbf{p}_u$ is the solution to a regularized least squares problem.
2. **Fix $P$, solve for $Q$**: Each item's embedding $\mathbf{q}_i$ is the solution to a regularized least squares problem.

Each step has a closed-form solution:
$$\mathbf{p}_u = (Q_u^T Q_u + \lambda I)^{-1} Q_u^T \mathbf{r}_u$$

**Why ALS over SGD?**
- Each subproblem is convex → guaranteed to improve
- Each user/item update is **independent** → trivially parallelizable across machines
- Handles implicit feedback naturally (Hu, Koren & Volinsky 2008)
- Preferred in distributed systems (Apache Spark MLlib uses ALS)

### Latent Factor Interpretation

What do the $k$ dimensions represent? For movie recommendations, factors tend to capture concepts like genre (action vs drama), era (80s vs 90s), tone (serious vs comedic), and audience (family vs adult). These aren't labeled — they emerge from the data. A user embedding $\mathbf{p}_u$ represents how much the user aligns with each latent factor. The dot product $\mathbf{p}_u \cdot \mathbf{q}_i$ measures overall alignment.

In [ ]:
%pip install -q numpy pandas scipy scikit-learn matplotlib tqdm

In [ ]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
import matplotlib.pyplot as plt
from tqdm import tqdm
import warnings, os, requests, zipfile, io, time

warnings.filterwarnings("ignore")
np.random.seed(42)
plt.style.use("seaborn-v0_8-whitegrid")

# ── Load MovieLens 100K ──────────────────────────────────────────────────────
DATA_DIR = "/tmp/ml-100k"
if not os.path.exists(DATA_DIR):
    r = requests.get("https://files.grouplens.org/datasets/movielens/ml-100k.zip", timeout=60)
    zipfile.ZipFile(io.BytesIO(r.content)).extractall("/tmp/")

ratings = pd.read_csv(f"{DATA_DIR}/u.data", sep="\t",
                      names=["user_id", "item_id", "rating", "timestamp"])
movies = pd.read_csv(f"{DATA_DIR}/u.item", sep="|", encoding="latin-1",
                     header=None, usecols=[0,1], names=["item_id", "title"])

n_users = ratings.user_id.max()
n_items = ratings.item_id.max()
print(f"Users: {n_users}, Items: {n_items}, Ratings: {len(ratings):,}")
print("Ready.")

## 1. Train/Test Split

We use a temporal split: hold out each user's most recent 20% of ratings for testing. This is more realistic than a random split — in production you always predict future behavior from past behavior.

**Why not random split?** Random splits leak future information into training. If a user rates a movie on Tuesday and another on Monday, a random split might train on Tuesday's rating and predict Monday's — that's time travel.

In [ ]:
# Temporal train/test split: hold out last 20% of each user's ratings
ratings_sorted = ratings.sort_values(["user_id", "timestamp"])
train_list, test_list = [], []

for uid, group in ratings_sorted.groupby("user_id"):
    n = len(group)
    cutoff = max(1, int(n * 0.8))
    train_list.append(group.iloc[:cutoff])
    test_list.append(group.iloc[cutoff:])

train = pd.concat(train_list).reset_index(drop=True)
test = pd.concat(test_list).reset_index(drop=True)

print(f"Train ratings: {len(train):,} ({len(train)/len(ratings):.1%})")
print(f"Test ratings:  {len(test):,} ({len(test)/len(ratings):.1%})")

# Build sparse train matrix
def build_sparse(df, n_users, n_items):
    rows = df.user_id.values - 1
    cols = df.item_id.values - 1
    data = df.rating.values.astype(np.float32)
    return csr_matrix((data, (rows, cols)), shape=(n_users, n_items))

R_train = build_sparse(train, n_users, n_items)
R_test  = build_sparse(test,  n_users, n_items)
print(f"Train matrix: {R_train.shape}, nnz={R_train.nnz:,}")

## 2. Funk SVD: SGD on Observed Entries

We implement Funk SVD from scratch with:
- Random initialization of user/item embeddings
- SGD updates over observed (user, item, rating) triples
- L2 regularization to prevent overfitting on sparse data
- Bias terms: global mean + user bias + item bias — account for the fact that some users always rate high (lenient) and some items are universally liked

$$\hat{r}_{ui} = \mu + b_u + b_i + \mathbf{p}_u \cdot \mathbf{q}_i$$

Bias terms alone explain ~25% of variance in the Netflix dataset. Always include them.

In [ ]:
class FunkSVD:
    """Funk SVD: Matrix factorization via SGD on observed entries only."""

    def __init__(self, n_factors=50, n_epochs=20, lr=0.005, reg=0.02,
                 min_rating=1.0, max_rating=5.0, verbose=True):
        self.k = n_factors
        self.n_epochs = n_epochs
        self.lr = lr
        self.reg = reg
        self.min_r = min_rating
        self.max_r = max_rating
        self.verbose = verbose

    def fit(self, train_df, n_users, n_items):
        self.n_users = n_users
        self.n_items = n_items

        # Initialize embeddings with small random values
        scale = 0.01
        self.P = np.random.normal(0, scale, (n_users, self.k)).astype(np.float32)  # user
        self.Q = np.random.normal(0, scale, (n_items, self.k)).astype(np.float32)  # item
        self.bu = np.zeros(n_users, dtype=np.float32)  # user biases
        self.bi = np.zeros(n_items, dtype=np.float32)  # item biases
        self.mu = train_df.rating.mean()  # global mean

        # Convert to arrays for fast access
        users = train_df.user_id.values - 1
        items = train_df.item_id.values - 1
        ratings_arr = train_df.rating.values.astype(np.float32)
        n = len(ratings_arr)

        train_losses, val_rmses = [], []

        for epoch in range(self.n_epochs):
            # Shuffle
            idx = np.random.permutation(n)
            users_s, items_s, ratings_s = users[idx], items[idx], ratings_arr[idx]

            epoch_loss = 0.0
            for u, i, r in zip(users_s, items_s, ratings_s):
                pred = self.mu + self.bu[u] + self.bi[i] + self.P[u] @ self.Q[i]
                err = r - pred
                epoch_loss += err ** 2

                # Update biases
                self.bu[u] += self.lr * (err - self.reg * self.bu[u])
                self.bi[i] += self.lr * (err - self.reg * self.bi[i])

                # Update embeddings
                pu_old = self.P[u].copy()
                self.P[u] += self.lr * (err * self.Q[i] - self.reg * self.P[u])
                self.Q[i] += self.lr * (err * pu_old  - self.reg * self.Q[i])

            rmse = np.sqrt(epoch_loss / n)
            train_losses.append(rmse)
            if self.verbose and (epoch + 1) % 5 == 0:
                print(f"Epoch {epoch+1:3d}/{self.n_epochs} | Train RMSE: {rmse:.4f}")

        self.train_losses = train_losses
        return self

    def predict(self, user_id, item_id):
        u, i = user_id - 1, item_id - 1
        pred = self.mu + self.bu[u] + self.bi[i] + self.P[u] @ self.Q[i]
        return float(np.clip(pred, self.min_r, self.max_r))

    def evaluate_rmse(self, test_df):
        errors = []
        for _, row in test_df.iterrows():
            pred = self.predict(int(row.user_id), int(row.item_id))
            errors.append((row.rating - pred) ** 2)
        return np.sqrt(np.mean(errors))


print("Training Funk SVD (k=50, 20 epochs)...")
t0 = time.time()
funk = FunkSVD(n_factors=50, n_epochs=20, lr=0.005, reg=0.02, verbose=True)
funk.fit(train, n_users, n_items)
print(f"\nTraining time: {time.time()-t0:.1f}s")
print(f"Test RMSE: {funk.evaluate_rmse(test):.4f}")

## 3. ALS: Alternating Least Squares

ALS alternates between solving for all user embeddings (holding items fixed) and all item embeddings (holding users fixed). Each user's embedding is the solution to an independent regularized least squares problem — so all users can be updated in parallel.

The closed-form update for user $u$:
$$\mathbf{p}_u = (Q_{I_u}^T Q_{I_u} + \lambda I)^{-1} Q_{I_u}^T \mathbf{r}_u^{I_u}$$

where $I_u$ is the set of items rated by user $u$. We solve this with `np.linalg.solve` which is numerically more stable than explicit matrix inversion.

In [ ]:
class ALS:
    """Alternating Least Squares matrix factorization."""

    def __init__(self, n_factors=50, n_iters=15, reg=0.1, verbose=True):
        self.k = n_factors
        self.n_iters = n_iters
        self.reg = reg
        self.verbose = verbose

    def fit(self, R_sparse, n_users, n_items):
        """R_sparse: csr_matrix of shape (n_users, n_items)."""
        self.n_users = n_users
        self.n_items = n_items

        # Initialize item embeddings randomly; user embeddings solved first
        self.P = np.random.normal(0, 0.01, (n_users, self.k)).astype(np.float64)
        self.Q = np.random.normal(0, 0.01, (n_items, self.k)).astype(np.float64)

        R_csr = R_sparse  # user-indexed rows
        R_csc = R_sparse.tocsc()  # item-indexed cols

        reg_eye = self.reg * np.eye(self.k)
        losses = []

        for it in range(self.n_iters):
            # ── Step 1: Fix Q, solve for each P[u] ──────────────────────────
            for u in range(n_users):
                items_u = R_csr[u].indices
                if len(items_u) == 0:
                    continue
                Q_u = self.Q[items_u]       # (n_u, k)
                r_u = np.array(R_csr[u, items_u].todense()).flatten()  # (n_u,)
                A = Q_u.T @ Q_u + reg_eye
                b = Q_u.T @ r_u
                self.P[u] = np.linalg.solve(A, b)

            # ── Step 2: Fix P, solve for each Q[i] ──────────────────────────
            for i in range(n_items):
                users_i = R_csc[:, i].indices
                if len(users_i) == 0:
                    continue
                P_i = self.P[users_i]       # (n_i, k)
                r_i = np.array(R_csc[users_i, i].todense()).flatten()  # (n_i,)
                A = P_i.T @ P_i + reg_eye
                b = P_i.T @ r_i
                self.Q[i] = np.linalg.solve(A, b)

            # Training loss
            loss = self._compute_rmse(R_csr)
            losses.append(loss)
            if self.verbose:
                print(f"Iter {it+1:2d}/{self.n_iters} | Train RMSE: {loss:.4f}")

        self.losses = losses
        return self

    def _compute_rmse(self, R_csr):
        errors = []
        cx = R_csr.tocoo()
        for u, i, r in zip(cx.row, cx.col, cx.data):
            pred = np.clip(self.P[u] @ self.Q[i], 1.0, 5.0)
            errors.append((r - pred) ** 2)
        return np.sqrt(np.mean(errors))

    def predict(self, user_id, item_id):
        pred = self.P[user_id-1] @ self.Q[item_id-1]
        return float(np.clip(pred, 1.0, 5.0))

    def evaluate_rmse(self, test_df):
        errors = []
        for _, row in test_df.iterrows():
            pred = self.predict(int(row.user_id), int(row.item_id))
            errors.append((row.rating - pred)**2)
        return np.sqrt(np.mean(errors))


print("Training ALS (k=50, 15 iterations)...")
t0 = time.time()
als = ALS(n_factors=50, n_iters=15, reg=0.1, verbose=True)
als.fit(R_train, n_users, n_items)
print(f"\nTraining time: {time.time()-t0:.1f}s")
print(f"Test RMSE: {als.evaluate_rmse(test):.4f}")

## 4. Latent Factor Visualization

One of the most compelling aspects of matrix factorization is that the latent dimensions sometimes align with interpretable concepts. Let's project movies into the 2D space defined by the first two ALS factors and see what clusters emerge.

In [ ]:
# Visualize item embeddings in 2D (first two latent factors)
fig, ax = plt.subplots(figsize=(10, 8))

# Get items with enough ratings for reliable embeddings
popular_items = ratings.groupby("item_id").size()
popular_items = popular_items[popular_items >= 50].index.tolist()[:80]

item_embeddings = als.Q[np.array(popular_items) - 1, :2]  # first 2 factors

ax.scatter(item_embeddings[:, 0], item_embeddings[:, 1],
           alpha=0.6, s=60, c="steelblue")

# Label a sample of movies
labeled = movies[movies.item_id.isin(popular_items[:30])].copy()
for _, row in labeled.iterrows():
    idx = popular_items.index(row.item_id)
    title = row.title[:25]
    ax.annotate(title, (item_embeddings[idx, 0], item_embeddings[idx, 1]),
                fontsize=7, alpha=0.8,
                xytext=(3, 3), textcoords="offset points")

ax.set_xlabel("Latent Factor 1")
ax.set_ylabel("Latent Factor 2")
ax.set_title("Item Embeddings: ALS Latent Space (first 2 factors)\nNearby movies are co-rated by similar users")
plt.tight_layout()
plt.savefig("/tmp/latent_factors.png", dpi=120)
plt.show()
print("Items close together in this space are recommended together")

## 5. Precision@K for Implicit Feedback

RMSE measures average rating prediction error — useful for explicit feedback. For implicit feedback (and ranking), we want **Precision@K**: of the top-K items we recommend, what fraction does the user actually interact with?

We binarize ratings ≥ 4 as "liked" and evaluate whether the model's top-K predictions include items the user liked in the test set.

In [ ]:
def precision_at_k(model_P, model_Q, test_df, train_df, K=10, n_eval_users=200):
    """Precision@K: fraction of top-K recs that are relevant (rating >= 4)."""
    precisions = []
    test_users = test_df[test_df.rating >= 4].user_id.unique()[:n_eval_users]

    for uid in test_users:
        # Items the user liked in test set
        liked_items = set(test_df[(test_df.user_id == uid) & (test_df.rating >= 4)].item_id)
        if not liked_items:
            continue

        # Items already seen in training
        seen_items = set(train_df[train_df.user_id == uid].item_id)

        # Score all unseen items
        u_idx = uid - 1
        scores = model_P[u_idx] @ model_Q.T  # shape: (n_items,)

        # Mask seen items
        for i in seen_items:
            if i - 1 < len(scores):
                scores[i - 1] = -np.inf

        top_k = np.argsort(scores)[::-1][:K] + 1  # +1 for 1-indexed item IDs
        hits = len(set(top_k) & liked_items)
        precisions.append(hits / K)

    return np.mean(precisions)

# Evaluate both models
p_funk_10 = precision_at_k(funk.P, funk.Q, test, train, K=10)
p_als_10  = precision_at_k(als.P,  als.Q,  test, train, K=10)
p_funk_20 = precision_at_k(funk.P, funk.Q, test, train, K=20)
p_als_20  = precision_at_k(als.P,  als.Q,  test, train, K=20)

results = pd.DataFrame({
    "Model": ["Funk SVD", "ALS"],
    "Test RMSE": [funk.evaluate_rmse(test), als.evaluate_rmse(test)],
    "Precision@10": [p_funk_10, p_als_10],
    "Precision@20": [p_funk_20, p_als_20],
})
print("\nModel Comparison:")
print(results.to_string(index=False))
print("\nNote: Lower RMSE ≠ better ranking. Both metrics matter depending on your goal.")

## 6. Convergence & Hyperparameter Sensitivity

Number of latent factors $k$ is the primary hyperparameter. Too small: underfitting — can't capture item nuance. Too large: overfitting on sparse data + slower inference. For most production systems, $k \in [64, 512]$ with regularization tuned on a validation set.

In [ ]:
# Compare training curves: Funk SGD vs ALS
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(range(1, len(funk.train_losses)+1), funk.train_losses,
             marker="o", markersize=4, label="Funk SGD", color="coral")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Train RMSE")
axes[0].set_title("Funk SGD Convergence")
axes[0].legend()

axes[1].plot(range(1, len(als.losses)+1), als.losses,
             marker="s", markersize=4, label="ALS", color="steelblue")
axes[1].set_xlabel("Iteration")
axes[1].set_ylabel("Train RMSE")
axes[1].set_title("ALS Convergence (faster per step, fewer iterations needed)")
axes[1].legend()

plt.tight_layout()
plt.savefig("/tmp/convergence.png", dpi=120)
plt.show()

# Hyperparameter sensitivity: k factors vs RMSE
print("\nTraining MF with different numbers of latent factors...")
factor_results = []
for k in [5, 10, 20, 50]:
    m = FunkSVD(n_factors=k, n_epochs=15, lr=0.005, reg=0.02, verbose=False)
    m.fit(train, n_users, n_items)
    rmse = m.evaluate_rmse(test)
    factor_results.append({"k": k, "test_rmse": rmse})
    print(f"  k={k:3d}: Test RMSE = {rmse:.4f}")

factor_df = pd.DataFrame(factor_results)
print(factor_df.to_string(index=False))

## Summary

### Key Concepts

| Concept | Key Point |
|---|---|
| Funk SVD | SGD on observed entries only — the Netflix Prize insight (2006) |
| Missing ≠ 0 | Treating unobserved ratings as zero is a critical mistake |
| Bias terms | $\mu + b_u + b_i$ explains ~25% of variance before any latent factors |
| ALS | Closed-form alternating updates — parallelizable, preferred for implicit |
| Regularization $\lambda$ | Controls overfitting; more important as sparsity increases |
| Latent factors | Emerge from data — capture genre, era, tone without labels |
| RMSE vs Precision@K | RMSE measures accuracy; Precision@K measures ranking — choose by your UX |
| Optimal $k$ | Usually 64–512 for production; diminishing returns beyond 256 |

### Common Pitfalls
- Skipping bias terms — they're simple and highly effective
- Using standard SVD (which requires complete matrix) instead of Funk SVD
- Tuning $k$ without also tuning $\lambda$ — they interact strongly
- Evaluating only on RMSE when the downstream task is ranking
- Not caching item embeddings at serving time — $O(n)$ per query, not $O(n \cdot k)$